In [128]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [129]:
with open('C:\\Users\\dylan\\OneDrive\\Documentos\\mensajeria\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [130]:
dim_proveedor=pd.read_sql_table('dim_proveedor', etl_conn)
dim_mensajero=pd.read_sql_table('dim_mensajero', etl_conn)
dim_sede=pd.read_sql_table('dim_sede', etl_conn)
dim_hora=pd.read_sql_table('dim_hora', etl_conn)    
dim_geografia=pd.read_sql_table('dim_geografia', etl_conn)
dim_fecha=pd.read_sql_table('dim_fecha', etl_conn)
dim_usuario=pd.read_sql_table('dim_usuario', etl_conn)
trans_servicio=pd.read_sql_table('trans_servicio', etl_conn)

tabla_sede=pd.read_sql_table('clientes_usuarioaquitoy', mensajeria)
tabla_estados=pd.read_sql_table('mensajeria_estadosservicio', mensajeria)

In [131]:
trans_servicio.head()


,servicio_id,fecha_solicitud,hora_solicitud,cliente_id,mensajero_id,usuario_id,ciudad_destino_id,ciudad_origen_id,mensajero2_id,mensajero3_id,...,fecha_iniciado,fecha_mensajero_asignado,fecha_recogido_mensajero,fecha_entregado,fecha_finalizado_completo,hora_iniciado,hora_mensajero_asignado,hora_recogido_mensajero,hora_entregado,hora_finalizado_completo
0,34,2023-10-26,09:46:03,5,0,10,1,1,0,0,...,2023-10-26,NaT,NaT,NaT,NaT,09:46:03,None,None,None,None
1,35,2023-10-26,11:18:14,5,7,8,1,1,0,0,...,2023-10-26,2023-10-28,2023-10-28,2023-12-07,NaT,11:18:14,14:43:08,19:45:18,01:25:34,None
2,36,2023-10-28,19:21:01,5,0,8,1,1,0,0,...,2023-10-28,NaT,NaT,NaT,NaT,19:21:01,None,None,None,None
3,41,2023-11-07,09:46:09,5,0,173,1,1,0,0,...,2023-11-07,NaT,NaT,NaT,NaT,09:46:09,None,None,None,None
4,42,2023-11-07,09:46:10,5,0,173,1,1,0,0,...,2023-11-07,NaT,NaT,NaT,NaT,09:46:10,None,None,None,None


In [143]:
hecho_servicios= trans_servicio.merge(dim_proveedor[['id_proveedor','key_proveedor']], left_on='cliente_id',right_on='id_proveedor', how='left') \
    .merge(dim_mensajero[['id_operaciones','key_mensajero']], left_on='mensajero_id',right_on='id_operaciones', how='left') \
    .merge(dim_mensajero[['id_operaciones','key_mensajero']], left_on='mensajero2_id', right_on='id_operaciones', how='left') \
    .merge(dim_mensajero[['id_operaciones','key_mensajero']], left_on='mensajero3_id', right_on='id_operaciones', how='left')  \
    .merge(dim_sede[['sede_id','key_sede']], left_on='sede_id', right_on='sede_id', how='left') \
    .merge(dim_geografia[['ciudad_id','key_geografia']], left_on='ciudad_origen_id',right_on='ciudad_id', how='left') \
    .merge(dim_geografia[['ciudad_id','key_geografia']], left_on='ciudad_destino_id',right_on='ciudad_id', how='left') \
    


#####Merge con dim_fecha y dim_hora para cada fecha y hora relevante en hecho_servicios
hecho_servicios= hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_solicitud', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora','key_hora']], left_on='hora_solicitud', right_on='hora', how='left') \
                                
hecho_servicios= hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_solicitud','key_hora':'key_dim_hora_solicitud'}, inplace=False)
print(hecho_servicios['key_dim_hora_solicitud'].value_counts())                   
                            
hecho_servicios= hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_iniciado', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_formato','key_hora']], left_on='hora_iniciado', right_on='hora_formato', how='left') \
                                
hecho_servicios= hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_iniciado','key_hora':'key_dim_hora_iniciado','fecha_completa_x':'fecha_completa'}, inplace=False)



hecho_servicios= hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_mensajero_asignado', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_formato','key_hora']], left_on='hora_mensajero_asignado', right_on='hora_formato', how='left') \

hecho_servicios= hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_mensajero_asignado','key_hora':'key_dim_hora_mensajero_asignado',
                                                 'fecha_completa_x':'fecha_completa'}, inplace=False)



hecho_servicios= hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_recogido_mensajero', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_formato','key_hora']], left_on='hora_recogido_mensajero', right_on='hora_formato', how='left') \


hecho_servicios= hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_recogido_mensajero','key_hora':'key_dim_hora_recogido_mensajero',
                                                 'fecha_completa_x':'fecha_completa','hora_formato_x':'hora_formato'}, inplace=False)
hecho_servicios.drop(columns=['fecha_completa_y','hora_formato_y'], inplace=True)                           


hecho_servicios= hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_entregado', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_formato','key_hora']], left_on='hora_entregado', right_on='hora_formato', how='left') \
                                
hecho_servicios= hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_entregado','key_hora':'key_dim_hora_entregado',
                                                 'fecha_completa_x':'fecha_completa','hora_formato_x':'hora_formato'}, inplace=False)
hecho_servicios.drop(columns=['fecha_completa_y','hora_formato_y'], inplace=True)


hecho_servicios= hecho_servicios.merge(dim_fecha[['fecha_completa','key_fecha']], left_on='fecha_finalizado_completo', right_on='fecha_completa', how='left') \
                                .merge(dim_hora[['hora_formato','key_hora']], left_on='hora_finalizado_completo', right_on='hora_formato', how='left') \

hecho_servicios= hecho_servicios.rename(columns={'key_fecha':'key_dim_fecha_finalizado_completo','key_hora':'key_dim_hora_finalizado_completo',
                                                 'fecha_completa_x':'fecha_completa','hora_formato_x':'hora_formato'}, inplace=False)
hecho_servicios.drop(columns=['fecha_completa_y','hora_formato_y'], inplace=True)


hecho_servicios.drop(columns=['id_proveedor','id_operaciones_x','id_operaciones_y','id_operaciones','ciudad_id_x','ciudad_id_y',
                              'mensajero_id','mensajero2_id','mensajero3_id','ciudad_origen_id','ciudad_destino_id','sede_id','cliente_id','usuario_id',
                                'user_id','hora','hora_entregado','hora_iniciado','servicio_id','fecha_solicitud','fecha_iniciado','hora_solicitud','hora_iniciado','fecha_entregado','fecha_finalizado_completo','hora_finalizado_completo',
                                'fecha_mensajero_asignado','fecha_recogido_mensajero',
                                'hora_mensajero_asignado','hora_recogido_mensajero'
                              ], inplace=True)


hecho_servicios.rename(columns={'key_proveedor':'key_dim_proveedor',
                                'key_mensajero_x':'key_dim_mensajero',
                                'key_mensajero_y':'key_dim_mensajero2',
                                'key_mensajero':'key_dim_mensajero3',
                                'key_geografia_x':'key_dim_geografia_origen',
                                'key_geografia_y':'key_dim_geografia_destino'
                                }, inplace=True)

hecho_servicios.info()

Series([], Name: count, dtype: int64)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28430 entries, 0 to 28429
Data columns (total 23 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   key_trans_servicio                 28430 non-null  int64         
 1   key_dim_proveedor                  28430 non-null  int64         
 2   key_dim_mensajero                  27703 non-null  float64       
 3   key_dim_mensajero2                 2233 non-null   float64       
 4   key_dim_mensajero3                 285 non-null    float64       
 5   key_sede                           3579 non-null   float64       
 6   key_dim_geografia_origen           28430 non-null  int64         
 7   key_dim_geografia_destino          28430 non-null  int64         
 8   fecha_completa                     28430 non-null  datetime64[ns]
 9   key_dim_fecha_solicitud            28430 non-null  int64         
 

In [140]:

print(hecho_servicios['key_dim_hora_iniciado'].value_counts())

Series([], Name: count, dtype: int64)


In [ ]:
hecho_servicios['key_dim_hora_mensajero_asignado'].value_counts()

Series([], Name: count, dtype: int64)

In [ ]:
# Compara los sets de valores únicos
  # si esto es 0 o muy bajo, ahí está el problema
# Compara valores "iguales" carácter por carácter
val_a = trans_servicio['hora_iniciado'].iloc[0]
val_b = dim_hora['hora_formato'].iloc[0]
print(repr(val_a), repr(val_b))   # repr muestra comillas, espacios ocultos, tipo exacto
print(type(val_a), type(val_b))

'09:46:03' '00:00:00'
<class 'str'> <class 'str'>
